# NB-11-02 — Retrieval, Prompting and Evaluation

This workbench accompanies L11-04, L11-05 and L11-07. It builds the RAG assistant over the
24-document handbook, compares retrievers, tests two prompt templates, and scores the results
at every stage.

In [ ]:
# Make the module's reference code importable from the notebooks/ directory.
import sys, pathlib
CODE_DIR = None
for _base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_base / "code").is_dir() and (_base / "notebooks").is_dir():
        CODE_DIR = _base / "code"
        break
if CODE_DIR is None:
    CODE_DIR = pathlib.Path.cwd().parent / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
print("code on path:", CODE_DIR)


In [ ]:
import sys
sys.path.insert(0, ".")

import numpy as np
from gen_data import QA, UNANSWERABLE, HANDBOOK
from rag import RAGSystem, evaluate_retrievers, build_context

system = RAGSystem()
print("documents:", len(system.docs), "chunks:", len(system.chunks))
print("embedding dim:", system.dense.embs.shape[1],
      "backend:", "faiss" if system.dense.faiss else "numpy")

## 1. Retrieval quality

Dense, lexical and hybrid retrieval on 30 labelled questions. BM25 wins at rank 1 because the
questions quote the handbook almost verbatim; fusion fixes the dense misses.

In [ ]:
res = evaluate_retrievers(system, QA)
print(f"{'retriever':16s} {'R@1':>6} {'R@3':>6} {'MRR':>6} {'nDCG@3':>7}")
for name, rep in res.items():
    print(f"{name:16s} {rep['recall@1']:6.2f} {rep['recall@3']:6.2f} "
          f"{rep['mrr']:6.3f} {rep['ndcg@3']:7.3f}")

## 2. One question through the pipeline

Watch the ranking change at each stage, then read the context the generator actually saw.

In [ ]:
q = "How much is the accommodation deposit and when is it refunded?"
for name, index in (("bm25", system.bm25), ("dense", system.dense), ("hybrid", system.hybrid)):
    hits = index.search(q, 3)
    print(f"{name:8s}", [(system.chunks[i].chunk_id, round(s, 2)) for i, s in hits])

out = system.answer(q)
print("answer:", out["answer"], "| docs:", out["docs"], "| support:", round(out["support"], 2))
print(out["context"][:300])

## 3. Prompts, measured

Two templates, six questions, one model. Few-shot costs twice the prompt tokens here; whether
it helps is a question for the data, not for intuition.

In [ ]:
from prompts import TEMPLATES, render, generate
from llm_eval import answer_report

for name in ("zero-shot", "few-shot"):
    preds = []
    for item in QA[:6]:
        p = render(TEMPLATES[name], HANDBOOK[item["source"]]["text"], item["q"])
        text, _ = generate(p)
        preds.append(text)
    rep = answer_report([(p, i["a"]) for p, i in zip(preds, QA[:6])])
    print(f"{name:10s} EM {rep['exact_match']:.2f} F1 {rep['token_f1']:.2f} "
          f"mean chars {np.mean([len(p) for p in preds]):.1f}")
    print("   ", preds[:3])

## 4. Evaluation at every stage

Retrieval, answers, grounding, refusal behaviour and cost — reported separately, because a
single end-to-end number hides which stage failed.

In [ ]:
from llm_eval import support_rate, unsupported_numbers, is_refusal

pairs, supports = [], []
for item in QA[:12]:
    o = system.answer(item["q"])
    pairs.append((o["answer"], item["a"]))
    supports.append(o["support"])
rep = answer_report(pairs)
print("EM", round(rep["exact_match"], 2), "F1", round(rep["token_f1"], 2),
      "support", round(float(np.mean(supports)), 2))

answers = [system.answer(q)["answer"] for q in UNANSWERABLE]
print("refusal rate on unanswerable:", round(np.mean([is_refusal(a) for a in answers]), 2))
for q, a in list(zip(UNANSWERABLE, answers))[:4]:
    print("   ", q[:44], "->", a)

ctx = HANDBOOK["hb-01"]["text"]
print("support:", round(support_rate("The library opens at 07:00.", ctx), 2),
      "unsupported numbers:", unsupported_numbers("The library opens at 07:00.", ctx))

## 5. Where it goes wrong

Two failures from the twelve questions above, both invisible to the retrieval metrics:
a right document with the wrong span, and a number from the wrong section.

In [ ]:
for item in QA[:8]:
    o = system.answer(item["q"])
    flag = "OK " if o["answer"].strip().lower() == item["a"].strip().lower() else "ERR"
    print(f"{flag} {item['q'][:48]:48s} -> {o['answer']!r} (gold {item['a']!r})")

## Exercises

1. Set the fusion weight `alpha` to 0.0 and 1.0 and explain the two endpoints.
2. Add a refusal threshold on the top retrieval score and report refusal and over-refusal.
3. Build five questions that need two documents and see whether retrieval finds both.
4. Replace MaxSim reranking with a cross-encoder and compare nDCG@3 and latency.